# Session 9 · Homework Solutions (Teacher Copy) — KNN Spam Filter

**Machine Learning Foundations · Sanketana School of Code**

Worked solution with commentary for the coach. The spam classes separate cleanly, so **all three k values land around 0.98–0.99 test accuracy** — a clean, balanced win where accuracy is a fair judge. Hold that thought: Sessions 12–13 break accuracy on *imbalanced* fraud data, and this honest result is the "before" picture.

**Acceptable variation:** any sensible k backed by its test number passes; scaling justified by "KNN uses distance" passes. Exact decimals shift a little with the split — grade the *reasoning and the number*, not the third decimal place.

## Step 1 · Load and look

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

spam = pd.read_csv("../../../datasets/secondary/spam_features.csv")
print("messages:", len(spam))
print("spam rate:", round(spam["is_spam"].mean(), 3))   # ~0.40, roughly balanced
spam.head()

## Step 2 · Scale, split, and fit KNN

Scaling matters because KNN ranks neighbours by distance and the raw features live on very different scales (a `message_length_words` of 150 would dwarf a `num_links` of 3 in raw distance). Fit the scaler on the training set only — the test set must stay unseen.

In [ ]:
feature_cols = [c for c in spam.columns if c != "is_spam"]
X = spam[feature_cols].values
y = spam["is_spam"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_train)          # training set only
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
print("test accuracy (k=5):", round(knn.score(X_test_s, y_test), 3))   # ~0.99

## Step 3 · Try three values of k

All three do well because spam and ham separate cleanly in feature space. Note k=1 scores a perfect **1.00 on training** — the memorising signature from class — yet its *test* score is still strong here because the classes barely overlap. On a harder problem that gap would be a red flag.

In [ ]:
for k in [1, 5, 15]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)
    tr = m.score(X_train_s, y_train)
    te = m.score(X_test_s, y_test)
    print(f"k={k:2d}: train {tr:.3f}  test {te:.3f}")

# Typical output:
#   k= 1: train 1.000  test ~0.995
#   k= 5: train ~0.992 test ~0.990
#   k=15: train ~0.987 test ~0.985

## Step 4 · ✅ Model answers

1. **Which k would you ship?** *"I'd ship k=5: it scores about 0.99 on the held-out test set, essentially tying k=1 and k=15, and a mid-range k is steadier than k=1 (which just memorises the training set) without over-smoothing like a very large k."* — any k in this range is defensible **as long as it cites the test number**. A bare "k=5" with no accuracy does **not** pass.

2. **Why scale first?** *"KNN classifies by distance to the nearest neighbours, so a feature with a large numeric range (like message length) would dominate 'closeness' and drown out the others. Scaling puts every feature on the same footing so the distance is fair."*

**Review talking point for Session 10:** KNN handed us a hard yes/no vote, but it never told us *how sure* it was about any single message. "How sure" — a probability between 0 and 1 — is exactly what logistic regression introduces next, along with why a straight line is the wrong tool for a yes/no question.